## RAG Pipelines- Data Ingestions to Vector DB Pipeline ##

### Read All Pdf's inside Data FOlder ###

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os

In [ ]:
def process_all_pdfs(directory):
    all_documents = []
    pdf_dir = Path(directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files Process..")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} Pages")
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nTotal Documents: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

In [ ]:
all_pdf_documents

In [35]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
        )

    splits_docs = text_spliter.split_documents(documents)
    print(f"splits {len(documents)} into {len(splits_docs)} chunks")

    if splits_docs:
        print(f"\nExample Chunk:")
        print(f"content: {splits_docs[0].page_content[:200]}")
        print(f"metadata: {splits_docs[0].metadata}")
    return splits_docs

In [ ]:
chunks = split_documents(all_pdf_documents)

### Embedding and VectorStoreDB ###

In [37]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple, Any
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded sucessfully. Embedding Dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not Loaded")
        print(f"Generating Embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embeddings with shape: {embeddings.shape}")
        return embeddings

embeddingmanager = EmbeddingManager()
embeddingmanager        

### Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = { "description":"PDF documents embeddings for RAG"}
            )

            print(f"Vector Store Initialized. Collection: {self.collection_name}")
            print(f"Existing Documents in collections: {self.collection.count()}")

        except Exception as e:
            print(f"Error Initializing the vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore = VectorStore()
vectorstore